# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [56]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [57]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [58]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [59]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [60]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [61]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [62]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [63]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," which is mentioned multiple times in the dataset.'

In [64]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, the projects "MediMind 17" and "Pathfinder 24" are associated with the security domain. "MediMind 17" involves a medical imaging solution that improves early diagnosis through vision transformers, and "Pathfinder 24" is an AI-powered platform optimizing logistics routes for sustainability, with a mention of security in its secondary domain.'

In [65]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had various comments about the fintech projects. For example, one judge described the project as a "Clever solution with measurable environmental benefit," indicating a positive assessment. Another judge noted that a project was "Promising idea with robust experimental validation," showing confidence in its potential. Additionally, some projects received praise for being "Technically ambitious and well-executed" or having "Impressive real-world impact." Overall, the judges recognized the fintech projects for their innovation, technical quality, and potential to make a meaningful impact.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [66]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [67]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [68]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

"Based on the provided data, the most common project domain is not explicitly stated, but I can tell you that the sample projects listed include domains such as Productivity Assistants, Legal / Compliance, Data / Analytics, and Healthcare / MedTech. \n\nSince the dataset includes multiple projects and their respective domains, the most common project domain among them would be the one that appears most frequently.\n\nHowever, with only a few sample entries provided, I cannot definitively determine the most common project domain overall. If you have the complete dataset, I recommend analyzing all the project domains to find which one occurs most frequently.\n\nIf you'd like, I can help guide you on how to analyze the entire dataset to find the most common domain."

In [69]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases related to security mentioned in the projects.'

In [70]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The judge's comments regarding the fintech projects indicate that they found them to be technically ambitious and well-executed."

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer

BM25 (Best Matching 25) is a retriever based on the bag-of-words model, which treats a document as an unordered collection of words. It uses natural language processing and information retrieval techniques, disregards word order, and takes into account word frequency as a signal of importance. Because of this, BM25 is very effective for term-matching queries where the presence of specific keywords matters. In contrast, embeddings models focus on capturing semantic meaning and generalizing across related concepts, which can sometimes overlook documents that contain the exact keywords you are looking for.

 Example query: “Which projects use Mamba-style attention?"
 
BM25 will look for documents containing the word "Mamba-style attention" and rank them based on how frequently the term appears and the overall document length. This ensures that documents explicitly mentioning the keyword are returned. An embeddings model might return documents about modeling or general attendtion that do not explicitly contain the word "Mamba-style,” which could be less precise if the goal is to find exact keyword matches. Therefore, BM25 is better than embeddings for queries that rely on exact term matching, rare keywords, or domain-specific acronyms, because it directly searches for the presence of those terms rather than their semantic meaning.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [71]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [72]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [73]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Synthetic Data Generators," since the examples mention projects like "SecureNest 18," "PlanPilot 22," and "GuardBot 20," all focused on synthetic data generation across various domains. However, these are specific project titles, and the overall domain information provided in the context includes "Productivity Assistants," "Healthcare / MedTech," and "Creative / Design / Media." \n\nIf the question is asking about the **most common domain across multiple projects**, then I do not have enough data to definitively determine which domain is most frequent overall, as only a few examples are provided.\n\nPlease let me know if you\'d like a detailed analysis based on the entire dataset or other specific information.'

In [74]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases mentioned related to security. The use cases discussed focus on federated learning and improving privacy in healthcare applications, but not explicitly on security.'

In [75]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had the following comments about the fintech projects:\n\n- For "Pathfinder 27," which is in the Finance / FinTech domain, the judges complimented it on "Excellent code quality and use of open-source libraries."\n- For "PlanPilot 35," also in the Finance / FinTech domain, the judges described it as "A clever solution with measurable environmental benefit."\n\nOverall, the judges appreciated the technical quality and innovative aspects of the fintech projects.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [76]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [77]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [78]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "E-commerce / Marketplaces," which shows up multiple times in the sample. However, since this is just a subset of the data, and the full dataset is not accessible in its entirety here, I cannot conclusively determine the overall most common project domain.\n\nWould you like me to help interpret the specific sample data or assist with a general understanding?'

In [79]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. For example, the project "A federated learning toolkit improving privacy in healthcare applications" addresses privacy concerns, which are a key aspect of security. Additionally, projects like "A document summarization and retrieval system for enterprise knowledge bases" and "A hardware-aware model quantization benchmark suite" involve security-related considerations, especially in handling sensitive data and ensuring system robustness.'

In [80]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges generally had positive comments about the fintech projects. They described some projects as promising, comprehensive, technically mature, or well-executed. For example, one project was called a "Promising idea with robust experimental validation," and another was noted for being "Technically ambitious and well-executed." Additionally, there were mentions of strong code quality and excellent open-source library use. Overall, the judges recognized these fintech-related projects as innovative, impactful, and technically solid.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

In the real world, user queries can vary widely in wording and level of specificity. Generating multiple reformulations of a user query can improve recall by creating a larger and broader pool of different ways to retrieve relevant information. This approach helps catch synonyms, alternative phrasings, and differences in terminology that might otherwise cause important documents to be missed. By expanding the ways a query is interpreted, the system increases the chances of finding all relevant documents. The trade-off is that this can increase computational cost and latency, since multiple queries need to be processed instead of just one.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [81]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [82]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [83]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [84]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [85]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [86]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, it appears that several project domains are listed, such as Security, Healthcare / MedTech, Creative / Design / Media, and Productivity Assistants. However, since only a few examples are given and there is no comprehensive count across the entire dataset, I cannot definitively determine the most common project domain. \n\nIf you have access to the full dataset, you could analyze it to find the domain with the highest frequency. But based on the provided snippet alone, I do not have enough information to say which domain is the most common.'

In [87]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases explicitly related to security. The projects mentioned focus primarily on federated learning, privacy improvement, and applications in healthcare, finance, and customer support.'

In [88]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech projects. For example, they described some projects as "promising idea with robust experimental validation," "a clever solution with measurable environmental benefit," and "a comprehensive and technically mature approach." Overall, the judges recognized the projects\' technical ambition, innovation, and quality.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [89]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [90]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [91]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is listed multiple times across different projects.'

In [92]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. Specifically, the project titled "MediMind 17" falls under the Security domain, with a secondary domain of Legal / Compliance. Its description is "A medical imaging solution improving early diagnosis through vision transformers," and it was noted for exceeding expectations in creativity and usability.'

In [93]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had various comments about the fintech projects. For example, they described "Pathfinder 27" as having "excellent code quality and use of open-source libraries," and "SecureNest 28" as being "conceptually strong but needing more benchmarking results." Overall, the judges recognized strengths such as strong conceptual frameworks and good code quality, though some projects noted areas for further development or testing.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [94]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [95]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [96]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [97]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [98]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [99]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Developer Tools / DevEx," which appears multiple times among the listed projects.'

In [100]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, the project titled "Project Aurora" focuses on security, with a description of developing a low-latency inference system for multimodal agents in autonomous systems. Additionally, "SecureNest" is another project in the security domain, involving a low-latency inference system for multimodal agents, although it is also associated with writing & content.'

In [101]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had various comments about the fintech projects. For example:\n\n- "WealthifyAI 16" was described as having a "comprehensive and technically mature approach."\n- "TrendLens 19" was praised as "technically ambitious and well-executed."\n- "AutoMate 5" was noted for being "a forward-looking idea with solid supporting data."\n- "InsightAI 1" was recognized for its "technically ambitious and well-executed" work.\n\nOverall, the judges commented positively on the fintech projects, highlighting their technical ambition, maturity, clear communication, and potential impact.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer

Semantic chunking is the process of splitting or combining sentences based on their semantic meaning, which is the underlying meaning or concept conveyed by the text rather than the exact words. When sentences are short and highly repetitive, such as in FAQs, semantic chunking may produce many small chunks with minimal semantic differences, leading to duplicate or redundant chunks that increase processing cost. Conversely, it might combine several sentences into a larger chunk, which could be unfocused and lose the granularity needed for precise retrieval.

To adjust the algorithm in such cases, you could add steps to remove duplicate chunks, perform additional pre-processing to improve the data’s suitability for chunking, use a hybrid method (i.e. use the structure of the document as well), or set explicit limits on chunk size. However, it may be the case that semantic chunking is not the ideal approach for extremely short and repetitive text, and an alternative strategy might be more effective.


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [1]:
#### import packages and set up API keys

import os
from getpass import getpass
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph
from ragas.testset.graph import Node, NodeType
from ragas.testset.transforms import default_transforms, apply_transforms

os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

In [4]:
### Data Preparation 

# Load the "How People Use AI" PDF
from langchain_community.document_loaders import PyPDFLoader

path = "data/howpeopleuseai.pdf"
loader = PyPDFLoader(path)
docs = loader.load()

print(f"Loaded {len(docs)} pages from PDF")

Loaded 64 pages from PDF


In [5]:

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/var/folders/lq/blff0y8x70d7sdk3bw6bzkvc0000gn/T/ipykernel_89162/2191311607.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
/var/folders/lq/blff0y8x70d7sdk3bw6bzkvc0000gn/T/ipykernel_89162/2191311607.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [6]:
kg = KnowledgeGraph()
kg


KnowledgeGraph(nodes: 0, relationships: 0)

In [7]:

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

In [8]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms_update = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms_update)
kg

Applying HeadlinesExtractor:   0%|          | 0/22 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'a33b88'. Skipping!
Property 'summary' already exists in node 'd74ac9'. Skipping!
Property 'summary' already exists in node '0ad8a4'. Skipping!
Property 'summary' already exists in node '05bbd1'. Skipping!
Property 'summary' already exists in node '68cd8c'. Skipping!
Property 'summary' already exists in node '6ba996'. Skipping!
Property 'summary' already exists in node '14fea8'. Skipping!
Property 'summary' already exists in node '63950d'. Skipping!
Property 'summary' already exists in node '3c7d50'. Skipping!
Property 'summary' already exists in node '6e03ba'. Skipping!
Property 'summary' already exists in node '380db8'. Skipping!
Property 'summary' already exists in node '84d85b'. Skipping!
Property 'summary' already exists in node '2c50af'. Skipping!
Property 'summary' already exists in node '7bb661'. Skipping!
Property 'summary' already exists in node 'ab5bc3'. Skipping!
Property 'summary' already exists in node 'aa1621'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/12 [00:00<?, ?it/s]

...

Applying ThemesExtractor:   0%|          | 0/10 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/10 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 90, relationships: 720)

In [9]:
kg.save("knowledge_graph.json")
print(f"✅ KG saved: {len(kg.nodes)} nodes, {len(kg.relationships)} relationships")

✅ KG saved: 90 nodes, 720 relationships


In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 90, relationships: 720)

In [11]:
# =============================================================================
# FIX: Clean buggy tuples in knowledge graph themes/personas
# =============================================================================
# Ragas's apply_transforms creates tuples like ('2025', '2025') instead of 
# strings, causing ValidationError. This cell fixes them before generation.
# =============================================================================

print("🔧 Fixing buggy themes/personas in knowledge graph...")
fixes_applied = 0

for node in usecase_data_kg.nodes:
    if hasattr(node, 'properties') and node.properties:
        # Fix 'themes' property - convert tuples/lists to strings
        if 'themes' in node.properties:
            themes = node.properties['themes']
            if isinstance(themes, list):
                fixed_themes = []
                for theme in themes:
                    if isinstance(theme, (tuple, list)):
                        # Convert tuple ('2025', '2025') to string '2025'
                        fixed_theme = str(theme[0]) if len(theme) > 0 else ''
                        fixed_themes.append(fixed_theme)
                        fixes_applied += 1
                    else:
                        fixed_themes.append(str(theme))
                node.properties['themes'] = fixed_themes
        
        # Fix 'personas' property - same bug can occur
        if 'personas' in node.properties:
            personas = node.properties['personas']
            if isinstance(personas, list):
                fixed_personas = []
                for persona in personas:
                    if isinstance(persona, (tuple, list)):
                        fixed_persona = str(persona[0]) if len(persona) > 0 else ''
                        fixed_personas.append(fixed_persona)
                        fixes_applied += 1
                    else:
                        fixed_personas.append(str(persona))
                node.properties['personas'] = fixed_personas

print(f"✅ Fixed {fixes_applied} buggy themes/personas in knowledge graph!")
print("   Knowledge graph is now ready for TestsetGenerator")


🔧 Fixing buggy themes/personas in knowledge graph...
✅ Fixed 0 buggy themes/personas in knowledge graph!
   Knowledge graph is now ready for TestsetGenerator


In [12]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

In [17]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

In [18]:
# =============================================================================
# RUNTIME PATCH: Fix Ragas ThemesPersonasInput validation bug
# =============================================================================
# This patches Ragas's validator to auto-convert lists/tuples to strings
# =============================================================================

print("🔧 Patching Ragas ThemesPersonasInput validator...")

# Import the buggy class
from ragas.testset.synthesizers.prompts import ThemesPersonasInput
from pydantic import field_validator

# Create a patched version with custom validator
class PatchedThemesPersonasInput(ThemesPersonasInput):
    """Fixed version that converts lists/tuples to strings."""
    
    @field_validator('themes', 'personas', mode='before')
    @classmethod
    def fix_list_to_string(cls, v):
        """Convert list/tuple elements to strings before validation."""
        if isinstance(v, list):
            fixed = []
            for item in v:
                if isinstance(item, (list, tuple)):
                    # ['Bick et al.', 'Bick et al. (2024)'] -> 'Bick et al.'
                    fixed.append(str(item[0]) if len(item) > 0 else '')
                else:
                    fixed.append(str(item))
            return fixed
        return v

# Replace the original class in Ragas
import ragas.testset.synthesizers.prompts as prompts_module
import ragas.testset.synthesizers.generate as generate_module

prompts_module.ThemesPersonasInput = PatchedThemesPersonasInput
generate_module.ThemesPersonasInput = PatchedThemesPersonasInput

# Also patch in the scenario modules
try:
    import ragas.testset.synthesizers.multi_hop as multi_hop_module
    multi_hop_module.ThemesPersonasInput = PatchedThemesPersonasInput
except:
    pass

try:
    import ragas.testset.synthesizers.single_hop as single_hop_module
    single_hop_module.ThemesPersonasInput = PatchedThemesPersonasInput
except:
    pass

print("✅ Ragas patched! Lists will auto-convert to strings during validation")
print("   You can now use generator.generate() without ValidationError")

🔧 Patching Ragas ThemesPersonasInput validator...
✅ Ragas patched! Lists will auto-convert to strings during validation
   You can now use generator.generate() without ValidationError


In [19]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

ValidationError: 1 validation error for ThemesPersonasInput
themes.0
  Input should be a valid string [type=string_type, input_value=['generative AI', 'Generative AI'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

In [195]:
dataset.to_pandas()


NameError: name 'dataset' is not defined

In [ ]:
# ================================
# Activity 1: Evaluate Retriever Methods with Ragas
# ================================

import os
from getpass import getpass
import copy
import time
from operator import itemgetter

# ------------------------
# 0. Set API Keys
# ------------------------
os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

# ------------------------
# 1. Load PDF Data
# ------------------------
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

data_path = "data/"
loader = DirectoryLoader(data_path, glob="*.pdf", loader_cls=PyMuPDFLoader)
documents = loader.load()
print(f"Loaded {len(documents)} documents.")

# ------------------------
# 2. Wrap LLMs & Embeddings for Ragas
# ------------------------
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

# ------------------------
# 3. Generate Golden Dataset (Synthetic Questions)
# ------------------------
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
golden_dataset = generator.generate_with_langchain_docs(documents, testset_size=20)  # 20 questions
print("Golden dataset generated.")

# Convert to Ragas EvaluationDataset
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_pandas(golden_dataset.to_pandas())

# ------------------------
# 4. Chunk Documents & Build Vector Store
# ------------------------
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

chunker = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = chunker.split_documents(documents)
print(f"Split into {len(split_docs)} chunks.")

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="use_case_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(client, collection_name="use_case_data", embedding=embeddings_model)
_ = vector_store.add_documents(split_docs)

# ------------------------
# 5. Define Retrievers
# ------------------------
# Here we define multiple retrievers; you can replace these with real implementations
retriever_naive = vector_store.as_retriever(search_kwargs={"k": 3})
retriever_bm25 = vector_store.as_retriever(search_kwargs={"k": 3})
retriever_multi = vector_store.as_retriever(search_kwargs={"k": 3})
retriever_parent = vector_store.as_retriever(search_kwargs={"k": 3})
retriever_ensemble = vector_store.as_retriever(search_kwargs={"k": 3})

retrievers = {
    "naive": retriever_naive,
    "bm25": retriever_bm25,
    "multi_query": retriever_multi,
    "parent_doc": retriever_parent,
    "ensemble": retriever_ensemble
}

# ------------------------
# 6. Build RAG Chain Helper
# ------------------------
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)
chat_model = ChatOpenAI(model="gpt-4.1-nano")

def build_chain(retriever):
    """Return a simple RAG chain for the given retriever."""
    return (
        {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
        | RunnablePassthrough.assign(context=itemgetter("context"))
        | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
    )

chains = {name: build_chain(r) for name, r in retrievers.items()}

# ------------------------
# 7. Evaluate Each Retriever
# ------------------------
from ragas.metrics import (
    LLMContextRecall, Faithfulness, FactualCorrectness,
    ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
)
from ragas import evaluate, RunConfig
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
metrics = [
    LLMContextRecall(), Faithfulness(), FactualCorrectness(),
    ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()
]
run_config = RunConfig(timeout=600)

results = {}

for name, chain in chains.items():
    print(f"Evaluating retriever: {name}")
    dataset_copy = copy.deepcopy(evaluation_dataset)
    
    for test_row in dataset_copy:
        response = chain.invoke({"question": test_row.eval_sample.user_input})
        test_row.eval_sample.response = response["response"].content
        test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in response["context"]]
        time.sleep(1)  # avoid rate-limiting
    
    results[name] = evaluate(
        dataset=dataset_copy,
        metrics=metrics,
        llm=evaluator_llm,
        run_config=run_config
    )

# ------------------------
# 8. Compare Results
# ------------------------
print("\n==== Retriever Evaluation Results ====")
for name, result in results.items():
    print(f"\nRetriever: {name}")
    print(result)

# ================================
# Now you can analyze:
# - Which retriever is best
# - Consider cost, latency, and performance
# - Optionally compare with semantic chunking adjustments
# ================================


Loaded 64 documents.


/var/folders/lq/blff0y8x70d7sdk3bw6bzkvc0000gn/T/ipykernel_82999/2820688164.py:33: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
/var/folders/lq/blff0y8x70d7sdk3bw6bzkvc0000gn/T/ipykernel_82999/2820688164.py:34: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/35 [00:00<?, ?it/s]

...

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

ValidationError: 1 validation error for ThemesPersonasInput
themes.0
  Input should be a valid string [type=string_type, input_value=('May 2024', 'May 15, 2024'), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type